In [17]:

import pandas as pd
import os 
import glob
from os import listdir
from os.path import isfile
from os.path import join
import numpy as np
from os import mkdir
from collections import Counter

In [18]:
def read_fasta(filename):
    """Reads sequences from a FASTA file."""
    sequences = []
    with open(filename, 'r') as file:
        for line in file:
            if not line.startswith('>'):
                sequences.append(line.strip())
    return sequences
def calculate_base_frequencies(sequences):
    """Calculates the frequency of each base at every position."""
    sequence_length = len(sequences[0])
    base_counts = [Counter() for _ in range(sequence_length)]

    for seq in sequences:
        for i, base in enumerate(seq):
            base_counts[i][base] += 1

    base_frequencies = []
    for counts in base_counts:
        total = sum(counts.values())
        frequencies = {base: count / total for base, count in counts.items()}
        base_frequencies.append(frequencies)
    return base_frequencies

# Subsampling from sequencing results(Short version)

In [25]:
os.makedirs('./short_subsampled_reads', exist_ok=True)  # create a new folder to save the subsampled reads
number_of_reads=[200,2000,10000,20000]
file="./P1_R1_001.fastq" # replace with the file from your Step8 folder
sequences=read_fasta(file)
for m in range(0,1):
    np.random.shuffle(sequences)
    #get the first  number_of_reads[i] sequences and save them to a new file
    for n in number_of_reads:
        with open(f'./short_subsampled_reads/short_P1_{n}reads_S{m+1}', 'w') as f:
            for seq in sequences[:n]:
                f.write(seq+'\n')

In [26]:
files = [f'./short_subsampled_reads/{f}' for f in listdir('./short_subsampled_reads/') if isfile(join('./short_subsampled_reads/', f)) and f.startswith('short_P1_')]
os.makedirs('./base_frequnceies', exist_ok=True)  # Ensure the output directory exists
base_frequencies={}
for file in files:
    sequences=read_fasta(file)
    base_frequencies[file] = calculate_base_frequencies(sequences)
    # save the base frequencies to an csv file,index is A, T, C, G and columns is position
    df = pd.DataFrame(base_frequencies[file]).fillna(0)
    df=df.T
    df.to_csv(f'./base_frequnceies/{os.path.basename(file)}_base_frequencies.csv', index=True)

# Subsampling from sequencing results(Extended version)

In [ ]:

# Define the list of FASTA files to process
fasta_files = [
    './S-mix-P1_AAAA_reads.fasta', 
    './S-mix-P1_AAAC_reads.fasta', 
    './S-mix-P1_AAAG_reads.fasta'
]# choose sample you want to use as refernce, move the output results 3 fasta files into the same folder as this script, and change the names of the files in the list above accordingly.

output_filename = './S-mix-P1_combined_payloads.txt'

with open(output_filename, 'w') as outfile:
    for file_path in fasta_files:
        # Check if the file exists before processing
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found. Skipping.")
            continue
            
        # Extract the index from the filename 
        # e.g., 'AAAA' from 'S-mix-P1_AAAA_reads.fasta'
        # Adjust the split logic if your actual filenames have a different structure
        index = os.path.basename(file_path).split('_')[1] 
        
        with open(file_path, 'r') as infile:
            sequence_buffer = []
            
            for line in infile:
                line = line.strip()
                # Skip header lines
                if line.startswith('>'):
                    # If we have a buffered sequence from the previous header, write it out
                    if sequence_buffer:
                        full_sequence = "".join(sequence_buffer)
                        outfile.write(f"{index}{full_sequence}\n")
                        sequence_buffer = [] # Reset buffer for the next sequence
                else:
                    # Collect sequence lines (handles multi-line sequences if they exist)
                    sequence_buffer.append(line)
            
            # Catch the very last sequence in the file after the loop finishes
            if sequence_buffer:
                full_sequence = "".join(sequence_buffer)
                outfile.write(f"{index}{full_sequence}\n")

print(f"Processing complete. Check '{output_filename}' for the combined sequences.")

Processing complete. Check './S-mix-P1_combined_payloads.txt' for the combined sequences.


In [ ]:
os.makedirs('./subsampled_reads', exist_ok=True)  # create a new folder to save the subsampled reads
number_of_reads = [200, 2000, 10000, 20000]  # change this to the number of reads you want to subsample
file="./S-mix-P1_combined_payloads.txt"
sequences=read_fasta(file)
for m in range(0,100):
    np.random.shuffle(sequences)
    #get the first  number_of_reads[i] sequences and save them to a new file
    for n in number_of_reads:
        with open(f'./subsampled_reads/Smix_P1_{n}reads_S{m+1}', 'w') as f:
            for seq in sequences[:n]:
                f.write(seq+'\n')


In [14]:


# Define the directory where your subsampled files are located
# Using '.' means the current directory. Adjust if they are in a specific folder.
input_directory = './subsampled_reads' 

# Find all files matching the subsample naming pattern (e.g., Smix_P1_50reads_S1.txt)
# Adjust the extension (.txt, .fasta, etc.) based on how you saved them
file_pattern = os.path.join(input_directory, "Smix_P1_*reads_S*") 
subsampled_files = glob.glob(file_pattern)

for file_path in subsampled_files:
    # Dictionary to group sequences by their 4-letter index
    groups = {}
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip empty lines or any remaining FASTA headers if they snuck in
            if not line or line.startswith('>'): 
                continue
            
            # Extract the 4-letter index and the actual DNA sequence
            index = line[:4]
            sequence = line[4:] 
            
            if index not in groups:
                groups[index] = []
            groups[index].append(sequence)
    
    # Skip creating an Excel file if the input file was empty
    if not groups:
        continue

    # Create an output Excel filename based on the input filename
    base_name = os.path.basename(file_path)
    file_prefix = os.path.splitext(base_name)[0]
    os.makedirs("./frequency_table", exist_ok=True)  # Ensure the output directory exists
    excel_filename = f"./frequency_table/{file_prefix}_frequencies.xlsx"
    
    # Write the calculated frequencies to an Excel file
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        for idx, seqs in groups.items():
            # Find the maximum sequence length in this group to define columns
            max_len = max(len(s) for s in seqs)
            
            # Initialize a dictionary to count A, C, G, T at each position
            counts = {
                'A': [0] * max_len, 
                'C': [0] * max_len, 
                'G': [0] * max_len, 
                'T': [0] * max_len
            }
            
            # Tally the nucleotides
            for seq in seqs:
                for pos, nuc in enumerate(seq):
                    nuc = nuc.upper()
                    if nuc in counts:
                        counts[nuc][pos] += 1
            
            # Convert counts to a pandas DataFrame
            # .T transposes the table so A, C, G, T become the rows
            df = pd.DataFrame(counts).T 
            
            # Label the columns as 1, 2, 3, etc.
            df.columns = [i + 1 for i in range(max_len)]
            
            # Save to a sheet named after the 4-letter index
            df.to_excel(writer, sheet_name=idx)
            
    print(f"Processed {base_name} -> Saved to {excel_filename}")

print("All subsampled files have been analyzed and exported.")

Processed Smix_P1_20000reads_S1 -> Saved to ./frequency_table/Smix_P1_20000reads_S1_frequencies.xlsx
Processed Smix_P1_10000reads_S2 -> Saved to ./frequency_table/Smix_P1_10000reads_S2_frequencies.xlsx
Processed Smix_P1_200reads_S2 -> Saved to ./frequency_table/Smix_P1_200reads_S2_frequencies.xlsx
Processed Smix_P1_10000reads_S1 -> Saved to ./frequency_table/Smix_P1_10000reads_S1_frequencies.xlsx
Processed Smix_P1_200reads_S1 -> Saved to ./frequency_table/Smix_P1_200reads_S1_frequencies.xlsx
Processed Smix_P1_2000reads_S1 -> Saved to ./frequency_table/Smix_P1_2000reads_S1_frequencies.xlsx
Processed Smix_P1_20000reads_S3 -> Saved to ./frequency_table/Smix_P1_20000reads_S3_frequencies.xlsx
Processed Smix_P1_2000reads_S2 -> Saved to ./frequency_table/Smix_P1_2000reads_S2_frequencies.xlsx
Processed Smix_P1_10000reads_S3 -> Saved to ./frequency_table/Smix_P1_10000reads_S3_frequencies.xlsx
Processed Smix_P1_20000reads_S2 -> Saved to ./frequency_table/Smix_P1_20000reads_S2_frequencies.xlsx
Pr